In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress

## Functions 

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def make_test_map(data,hdr,vmin=-0.5,vmax=0.5,cmap='RdBu_r',
                  *args,**kwargs):

    wcs = WCS(hdr)
    fs = 20
    plt.figure(figsize=(40,4))
    plt.subplot(projection=wcs)
    plt.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    plt.title(hdr['OBJECT'],fontsize=fs)
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs)

    gc.collect()

    return

In [ ]:
def make_kernel(fwhm,hdr):
    
    print('FWHM of kernel: '+str(fwhm)+' arcmin')   
    pixsize = hdr[0]['CDELT2']*60
    print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    print('standard deviation of kernel: '+str(stddev)+' pixels')

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    center = ((kernel.shape[0]-1)/2,(kernel.shape[0]-1)/2)
    circle1 = Circle(center, stddev, fill=False)    
    fig,ax = plt.subplots(1,1,figsize=(3,3))
    plt.imshow(kernel, interpolation='none', origin='lower')
    plt.xlabel('lon [pixels]')
    plt.ylabel('lat [pixels]')
    ax.add_patch(circle1)
    ax.set_aspect('equal')
    plt.show()
    
    return kernel,stddev

In [ ]:
def convolve_data(kernel,stddev,data_list,hdr_list,dir_out,
                  keep_conv=True,savefile=False,
                  regrid_data=False,pixfactor=1,*args,**kwargs):
    
    data_conv = []
    hdr_conv = []
    descrptn = []
    
    for i in range(0,len(data_list)):
            
        data_conv.append(convolve(data_list[i],kernel))
            
        if regrid_data:
            descrptn.append(hdr_list[i]['OBJECT']+'_conv'+str(int(stddev))+'_regrd')
        else:
            descrptn.append(hdr_list[i]['OBJECT']+'_conv'+str(int(stddev)))           
        print(descrptn[i])
                
        hdr_new = hdr_list[i].copy()
        hdr_new['OBJECT'] = descrptn[i]
        hdr_conv.append(hdr_new)
            
        print('finished convolving array '+str(i+1)+' of '+str(len(data_list)))
    
    if regrid_data:
        data_conv,hdr_conv = regrid_maps(pixfactor,data_conv,hdr_conv)

    if savefile:
        for i in range(0,len(data_list)):
            fits.writeto(dir_out+descrptn[i]+'.fits',data_conv[i],
                         header=hdr_conv[i],overwrite=True)

    if keep_conv == False:
        del data_conv
        data_conv = 0
    gc.collect()
    
    return data_conv,hdr_conv

In [ ]:
def regrid_maps(pixfactor,data_list,hdr_list):
    
    data_regrid = []
    hdr_regrid = []
    
    for i in range(0,len(data_list)):
        
        hdr_new = hdr_list[i].copy()
        hdr_new['CDELT1'] = hdr_new['CDELT1']*pixfactor
        hdr_new['CDELT2'] = hdr_new['CDELT2']*pixfactor
        hdr_new['NAXIS1'] = int(hdr_new['NAXIS1']/pixfactor)
        hdr_new['NAXIS2'] = int(hdr_new['NAXIS2']/pixfactor)
        hdr_new['CRPIX1'] = int(hdr_new['CRPIX1']/pixfactor)
        hdr_new['CRPIX2'] = int(hdr_new['CRPIX2']/pixfactor)

        hdr_regrid.append(hdr_new)
        data_regrid.append(reproject_interp((data_list[i],hdr_list[i]),hdr_new)[0])
        print('finished regridding array '+str(i+1)+' of '+str(len(data_list)))
            
    return data_regrid, hdr_regrid

## Convolve and regrid CGPS-only (C) and CGPS+GMIMS (CG) Stokes Q and U maps

In [ ]:
go      = True
dir_in  = '/srv/data/cgps-gmims/raw_QU/'
dir_out = '/srv/data/cgps-gmims/conv_regrid/
fwhm    = 3 # arcmin

In [ ]:
if go:

    CG_Q_list, hdr_CG_Q_list = read_files_4channels(dir_in,'Q','CG')
    CG_U_list, hdr_CG_U_list = read_files_4channels(dir_in,'U','CG')

    kernel,stddev = make_kernel(fwhm,hdr_CG_Q_list)

    CG_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,CG_Q_list,hdr_CG_Q_list,dir_out,
                                                 keep_conv=False, savefile=True,
                                                 regrid_data=True, pixfactor=2)

    CG_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,CG_U_list,hdr_CG_U_list,dir_out,
                                                  keep_conv=False, savefile=True,
                                                  regrid_data=True, pixfactor=2)


In [ ]:
if go:
    
    C_Q_list, hdr_C_Q_list = read_files_4channels(dir_in,'Q','C')
    C_U_list, hdr_C_U_list = read_files_4channels(dir_in,'U','C')

    kernel,stddev = make_kernel(fwhm,hdr_C_Q_list)

    C_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,C_Q_list,hdr_C_Q_list,dir_out,
                                                 keep_conv=False, savefile=True,
                                                 regrid_data=True, pixfactor=2)

    C_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,C_U_list,hdr_C_U_list,dir_out,
                                                  keep_conv=False, savefile=True,
                                                  regrid_data=True, pixfactor=2)


## Regrid GMIMS to same gridding as convolved regridded CGPS and CGPS+GMIMS files

In [ ]:
go      = True
dir_in  = '/srv/data/cgps-gmims/gmims_OG/'
dir_out = '/srv/data/cgps-gmims/conv_regrid/'

In [ ]:
if go:

    # Read in GMIMS data:
    G_Q_list, hdr_G_Q_list = read_files_4channels(dir_in,'Q','GMIMS_W21_HR')
    G_U_list, hdr_G_U_list = read_files_4channels(dir_in,'U','GMIMS_W21_HR')

    # Make the new headers based on previous regridded files:
    hdu_new = fits.open('/srv/data/cgps-gmims/conv_regrid/QA_CG_conv4_regrd.fits')
    hdr_new = hdu_new[0].header
    hdrQ_list = [hdr_new.copy(),hdr_new.copy(),hdr_new.copy(),hdr_new.copy()]
    hdrU_list = [hdr_new.copy(),hdr_new.copy(),hdr_new.copy(),hdr_new.copy()]
    hdrQ_list[0]['OBJECT'] = 'QA_G_regrd'
    hdrQ_list[1]['OBJECT'] = 'QB_G_regrd'
    hdrQ_list[2]['OBJECT'] = 'QC_G_regrd'
    hdrQ_list[3]['OBJECT'] = 'QD_G_regrd'
    hdrU_list[0]['OBJECT'] = 'UA_G_regrd'
    hdrU_list[1]['OBJECT'] = 'UB_G_regrd'
    hdrU_list[2]['OBJECT'] = 'UC_G_regrd'
    hdrU_list[3]['OBJECT'] = 'UD_G_regrd'

    # Do the regridding:
    for i in range(0,4):

        print(i)
    
        Q_G = reproject_interp((G_Q_list[i],hdr_G_Q_list[i]),hdrQ_list[i])[0]
        fits.writeto(dir_out+hdrQ_list[i]['OBJECT']+'.fits',
                     Q_G, header=hdrQ_list[i], overwrite=True)

        U_G = reproject_interp((G_U_list[i],hdr_G_U_list[i]),hdrU_list[i])[0]
        fits.writeto(dir_out+hdrU_list[i]['OBJECT']+'.fits',
                     U_G, header=hdrU_list[i], overwrite=True)
    

## Read in 'raw' CG files, convolve, regrid, calculate PA (Aug 22, 2023)

In [ ]:
%%time
Aug22_attempt = False

if Aug22_attempt:

    directory = '/srv/aordog/cgps_gmims_data/'

    CG_Q_list, hdr_CG_Q_list = read_files_4channels(directory,'Q','CG')
    CG_U_list, hdr_CG_U_list = read_files_4channels(directory,'U','CG')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_CG_Q_list)


    CG_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,CG_Q_list,hdr_CG_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    CG_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,CG_U_list,hdr_CG_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(CG_U_conv_list[0].shape)
    print(CG_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(CG_Q_conv_list,CG_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')


## Start from CG PA and calculate RM along with fitting parameters (Aug 23, 2023)

In [ ]:
%%time
Aug23_attempt = False

if Aug23_attempt:

    directory = '/srv/aordog/cgps_gmims_data/'
    
    PA_list, hdrPA_list = read_files_4channels(directory,'PA_','CG_conv4_regrd')
    print(PA_list[0].shape)

    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Read in 'raw' C files, convolve, regrid, calculate PA and RM (Aug 23, 2023)

In [ ]:
%%time
Aug23_C = False

if Aug23_C:

    directory = '/srv/aordog/cgps_gmims_data/'

    C_Q_list, hdr_C_Q_list = read_files_4channels(directory,'Q','C')
    C_U_list, hdr_C_U_list = read_files_4channels(directory,'U','C')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_C_Q_list)


    C_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,C_Q_list,hdr_C_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    C_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,C_U_list,hdr_C_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(C_U_conv_list[0].shape)
    print(C_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(C_Q_conv_list,C_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')
    
    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Read in 'raw' G files, convolve, regrid, calculate PA and RM (Aug 23, 2023)

In [ ]:
%%time
Aug23_G = False

if Aug23_G:

    directory = '/srv/aordog/cgps_gmims_data/'

    G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','G')
    G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','G')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_G_Q_list)


    G_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,G_Q_list,hdr_G_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    G_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,G_U_list,hdr_G_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(G_U_conv_list[0].shape)
    print(G_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(G_Q_conv_list,G_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')
    
    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Calculate PI for convolved and gridded data

In [ ]:
%%time
Aug23_PI = True

if Aug23_PI:

    directory = '/srv/aordog/cgps_gmims_data/'

    G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','G_conv4_regrd')
    G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','G_conv4_regrd')
    
    C_Q_list, hdr_C_Q_list = read_files_4channels(directory,'Q','C_conv4_regrd')
    C_U_list, hdr_C_U_list = read_files_4channels(directory,'U','C_conv4_regrd')
    
    CG_Q_list, hdr_CG_Q_list = read_files_4channels(directory,'Q','CG_conv4_regrd')
    CG_U_list, hdr_CG_U_list = read_files_4channels(directory,'U','CG_conv4_regrd')
    
    G_avg_PI, hdr_G_avg_PI = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='avg_PI',keep_PI=True,savefile=True)
    G_PI_avg, hdr_G_PI_avg = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='PI_avg',keep_PI=True,savefile=True)
    
    C_avg_PI, hdr_C_avg_PI = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='avg_PI',keep_PI=True,savefile=True)
    C_PI_avg, hdr_C_PI_avg = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='PI_avg',keep_PI=True,savefile=True)
    
    CG_avg_PI, hdr_CG_avg_PI = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='avg_PI',keep_PI=True,savefile=True)
    CG_PI_avg, hdr_CG_PI_avg = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='PI_avg',keep_PI=True,savefile=True)
    

## Simpler way to get regridded GMIMS:

In [ ]:
directory = '/srv/aordog/cgps_gmims_data/'

G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','GMIMS_W21_HR')
G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','GMIMS_W21_HR')

make_test_map(G_Q_list[0],hdr_G_Q_list[0],vmin=-1,vmax=1,cmap='RdBu_r')

In [ ]:
hdu_new = fits.open('/srv/aordog/cgps_gmims_data/G_old/QA_G_conv4_regrd.fits')
hdr_new = hdu_new[0].header
hdrQ_list = [hdr_new.copy(),hdr_new.copy(),hdr_new.copy(),hdr_new.copy()]
hdrU_list = [hdr_new.copy(),hdr_new.copy(),hdr_new.copy(),hdr_new.copy()]

hdrQ_list[0]['OBJECT'] = 'QA_G_conv4_regrd'
hdrQ_list[1]['OBJECT'] = 'QB_G_conv4_regrd'
hdrQ_list[2]['OBJECT'] = 'QC_G_conv4_regrd'
hdrQ_list[3]['OBJECT'] = 'QD_G_conv4_regrd'
hdrU_list[0]['OBJECT'] = 'UA_G_conv4_regrd'
hdrU_list[1]['OBJECT'] = 'UB_G_conv4_regrd'
hdrU_list[2]['OBJECT'] = 'UC_G_conv4_regrd'
hdrU_list[3]['OBJECT'] = 'UD_G_conv4_regrd'


In [ ]:
QA_G = reproject_interp((G_Q_list[0],hdr_G_Q_list[0]),hdrQ_list[0])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/QA_G_regrd.fits',QA_G,header=hdrQ_list[0],overwrite=True)

QB_G = reproject_interp((G_Q_list[1],hdr_G_Q_list[1]),hdrQ_list[1])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/QB_G_regrd.fits',QB_G,header=hdrQ_list[1],overwrite=True)

QC_G = reproject_interp((G_Q_list[2],hdr_G_Q_list[2]),hdrQ_list[2])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/QC_G_regrd.fits',QC_G,header=hdrQ_list[2],overwrite=True)

QD_G = reproject_interp((G_Q_list[3],hdr_G_Q_list[3]),hdrQ_list[3])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/QD_G_regrd.fits',QD_G,header=hdrQ_list[3],overwrite=True)


UA_G = reproject_interp((G_U_list[0],hdr_G_U_list[0]),hdrU_list[0])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/UA_G_regrd.fits',UA_G,header=hdrU_list[0],overwrite=True)

UB_G = reproject_interp((G_U_list[1],hdr_G_U_list[1]),hdrU_list[1])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/UB_G_regrd.fits',UB_G,header=hdrU_list[1],overwrite=True)

UC_G = reproject_interp((G_U_list[2],hdr_G_U_list[2]),hdrU_list[2])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/UC_G_regrd.fits',UC_G,header=hdrU_list[2],overwrite=True)

UD_G = reproject_interp((G_U_list[3],hdr_G_U_list[3]),hdrU_list[3])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/UD_G_regrd.fits',UD_G,header=hdrU_list[3],overwrite=True)

In [ ]:
phi_peak_regrd = reproject_interp((phi_peak,hdr_GMIMS_new),hdrU_list[3])[0]
fits.writeto('/srv/aordog/cgps_gmims_data/phi_peak_regrd.fits',phi_peak_regrd,header=hdrU_list[3],overwrite=True)

In [ ]:
make_test_map(QB_G,hdrQ_list[1],vmin=-1,vmax=1,cmap='RdBu_r')
make_test_map(UB_G,hdrU_list[1],vmin=-1,vmax=1,cmap='RdBu_r')

In [ ]:
G_Q_list = [QA_G, QB_G, QC_G, QD_G]
G_U_list = [UA_G, UB_G, UC_G, UD_G]

PA_list, hdrPA_list = calc_pol_angles(G_Q_list,G_U_list,hdrQ_list,hdrU_list,
                                      keep_PA=True,savefile=True)
    
make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')

RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)

make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

In [ ]:
directory = '/srv/aordog/cgps_gmims_data/'

G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','G_regrd')
G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','G_regrd')

G_avg_PI, hdr_G_avg_PI = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                       avg_order='avg_PI',keep_PI=True,savefile=True)
G_PI_avg, hdr_G_PI_avg = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                       avg_order='PI_avg',keep_PI=True,savefile=True)

In [ ]:
hdu_G_new = fits.open('/srv/aordog/cgps_gmims_data/RM_G_conv4_regrd.fits')
hdu_G_old = fits.open('/srv/aordog/cgps_gmims_data/G_old/RM_G_conv4_regrd.fits')

hdr_G_old = hdu_G_old[0].header
hdr_G_new = hdu_G_new[0].header

G_old = hdu_G_old[0].data
G_new = hdu_G_new[0].data

print(G_old.shape)
print(G_new.shape)

print(repr(hdr_G_old))
print('')
print(repr(hdr_G_new))

In [ ]:
make_test_map(G_old,hdr_G_old,vmin=-100,vmax=100,cmap='RdBu_r')
make_test_map(G_new,hdr_G_new,vmin=-100,vmax=100,cmap='RdBu_r')

In [ ]:
make_test_map(G_new-G_old,hdr_G_new,vmin=-1,vmax=1,cmap='RdBu_r')

## RM synthesis on GMIMS

In [ ]:
hdu_GMIMS = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')
hdr_GMIMS = hdu_GMIMS[0].header
#print(repr(hdr_GMIMS))

Q_GMIMS = hdu_GMIMS[0].data[1]
U_GMIMS = hdu_GMIMS[0].data[2]

print(Q_GMIMS.shape)
print(U_GMIMS.shape)

hdr_GMIMS_new = hdr_GMIMS.copy()
hdr_GMIMS_new['NAXIS'] = 3
hdr_GMIMS_new['WCSAXES'] = 3
del hdr_GMIMS_new['NAXIS4']
del hdr_GMIMS_new['CRPIX4']
del hdr_GMIMS_new['CDELT4']
del hdr_GMIMS_new['CTYPE4']
del hdr_GMIMS_new['CRVAL4']
#print(repr(hdr_GMIMS_new))

freq = np.linspace(hdr_GMIMS_new['CRVAL3'],
                   hdr_GMIMS_new['CRVAL3']+hdr_GMIMS_new['CDELT3']*(hdr_GMIMS_new['NAXIS3']-1),
                   hdr_GMIMS_new['NAXIS3'])

print(len(freq))
print(hdr_GMIMS_new['NAXIS3'])
print(freq[100]-freq[99])
print(hdr_GMIMS_new['CDELT3'])

#with open('/srv/aordog/cgps_gmims_data/gmims_freq.txt', 'w') as file:
#    file.write('\n'.join(map(str, freq)))

#fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_Q.fits',Q_GMIMS,header=hdr_GMIMS_new,overwrite=True)
#fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_U.fits',U_GMIMS,header=hdr_GMIMS_new,overwrite=True)

In [ ]:
hdu_GMIMS = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
hdr_GMIMS = hdu_GMIMS[0].header
#print(repr(hdr_GMIMS))

FDF_GMIMS = hdu_GMIMS[0].data

print(FDF_GMIMS.shape)

hdr_GMIMS_new = hdr_GMIMS.copy()
hdr_GMIMS_new['NAXIS'] = 2
hdr_GMIMS_new['WCSAXES'] = 2
del hdr_GMIMS_new['NAXIS3']
del hdr_GMIMS_new['CRPIX3']
del hdr_GMIMS_new['CDELT3']
del hdr_GMIMS_new['CTYPE3']
del hdr_GMIMS_new['CRVAL3']
del hdr_GMIMS_new['CUNIT3']
#print(repr(hdr_GMIMS_new))

phi = np.linspace(hdr_GMIMS['CRVAL3'],
                   hdr_GMIMS['CRVAL3']+hdr_GMIMS['CDELT3']*(hdr_GMIMS['NAXIS3']-1),
                   hdr_GMIMS['NAXIS3'])

#print(phi)

i_phi = np.nanmax(FDF_GMIMS,axis=0)
this = np.where(FDF_GMIMS==i_phi)[0]
print(this.shape)

phi_peak = np.empty_like(FDF_GMIMS[0])
print(phi_peak.shape)

for i in range(0,FDF_GMIMS.shape[1]):
    #print(i)
    for j in range(0,FDF_GMIMS.shape[2]):

        idx = np.where(FDF_GMIMS[:,i,j] == np.nanmax(FDF_GMIMS[:,i,j]))[0][0]
        #print(idx)
        phi_peak[i,j] = phi[idx]


In [ ]:
plt.imshow(phi_peak,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')

In [ ]:
fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_peak.fits',phi_peak,header=hdr_GMIMS_new,overwrite=True)

In [ ]:

freq = np.array([1406.9,1413.8,1427.4,1434.3])
lbd2 = ((3e8)/(freq*1e6))**2

print((lbd2[0]-lbd2[1])*100*180/np.pi)

print((2.5*np.pi/180)/(lbd2[0]-lbd2[1]))

In [ ]:
720*1440*201